# PyTorch Core Operations
Learning tensors through the lit-review-assistant data.

**Goal:** understand shapes, indexing, broadcasting, reductions, and matrix ops using real abstracts.

In [57]:
from pathlib import Path
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_PATH = Path('..') / 'data' / 'pubmed_results.csv'
df = pd.read_csv(DATA_PATH)
abstracts = df["abstract"].fillna("").tolist()
print(f"Loaded {len(abstracts)} abstracts")

Loaded 100 abstracts


## 1. Shapes and Dimensions
A tensor's `.shape` tells you its size in each dimension.
- `shape[0]` = number of rows (papers)
- `shape[1]` = number of columns (vocabulary features)

In [3]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
X = vectorizer.fit_transform(abstracts)
X_tensor = torch.tensor(X.toarray(), dtype=torch.float32)

print('Shape of tensor: ', X_tensor.shape) # (num_papers, vocab_size)
print('Dims: ', X_tensor.ndim) #2
print('Dtype: ', X_tensor.dtype) #torch.float32
print('Num papers: ', X_tensor.shape[0]) #100
print('Vocab size: ', X_tensor.shape[1]) #1000

Shape of tensor:  torch.Size([100, 1000])
Dims:  2
Dtype:  torch.float32
Num papers:  100
Vocab size:  1000


## 2. L2 Normalisation — why sum ≠ 1.0 but norm = 1.0

Two different normalisations exist:
- **L1**: divide by sum → values sum to 1.0
- **L2**: divide by √(sum of squares) → vector *length* = 1.0

We need L2 because cosine similarity = dot(a,b) / (|a|×|b|).
If |a|=|b|=1, that simplifies to just dot(a,b) — which is what `@` computes.

In [8]:
norms = X_tensor.norm(dim=1, keepdim=True).clamp(min=1e-10)
X_normalized = X_tensor / norms

print('Sum of first paper vector (L2): ', X_normalized[0].sum().item()) #NOT 1.0
print('Norm of first paper vector (L2): ', X_normalized[0].norm().item()) #1.0

#Compare with L1  normalization
X_l1 = X_tensor / X_tensor.sum(dim=1, keepdim=True).clamp(min=1e-8)
print('Sum of first paper vector (L1): ', X_l1[0].sum().item()) #1.0
print('Norm of first paper vector (L1): ', X_l1[0].norm().item()) #NOT 1.0

Sum of first paper vector (L2):  7.489206790924072
Norm of first paper vector (L2):  1.0000001192092896
Sum of first paper vector (L1):  0.9999998807907104
Norm of first paper vector (L1):  0.13352550566196442


## 3. Indexing and Slicing
Tensors index just like Python lists and Numpy arrays. Key difference: indexing a 2D tensor with one index returns a 1D tensor (a row), not a scalar.

In [13]:
sim_matrix = X_normalized @ X_normalized.T

print('Similarity matrix shape: ', sim_matrix.shape) #(num_papers, num_papers)

print('--- Indexing ---')

print('sim_matrix[0] shape: ', sim_matrix[0].shape) # 1D: row 0
print('sim_matrix[0, 1]: ', sim_matrix[0, 1].item()) # scalar: paper 0 vs 1
print('sim_matrix[:3, :3]:') # 3x3 block
print(sim_matrix[:3, :3])

print('--- Diagonal (self-similarity) ---')
print(sim_matrix.diagonal()[:5])  # should all be ~1.0 

Similarity matrix shape:  torch.Size([100, 100])
--- Indexing ---
sim_matrix[0] shape:  torch.Size([100])
sim_matrix[0, 1]:  0.28847748041152954
sim_matrix[:3, :3]:
tensor([[1.0000, 0.2885, 0.1476],
        [0.2885, 1.0000, 0.1021],
        [0.1476, 0.1021, 1.0000]])
--- Diagonal (self-similarity) ---
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 4. Reductions and the `dim` argument

The rule:
- `dim=0` collapses **across rows** (result has shape of a single row)
- `dim=1` collapses **across columns** (result has shape of a single column)

Think of it as: *which axis disappears after the operation?*

In [16]:
print('Total sum (scalar): ', X_tensor.sum().item())
print('Sum per vocab word (dim=0): ', X_tensor.sum(dim=0).shape)   # (1000,)
print('Sum per paper (dim=1):      ', X_tensor.sum(dim=1).shape)   # (N,)

# Which paper uses the most vocabulary (highest total TF-IDF weight)?
most_vocab_idx = X_tensor.sum(dim=1).argmax().item()
print(f'Paper with most vocab coverage: [{most_vocab_idx}] {df["title"][most_vocab_idx]}')

# Which paper uses the least vocabulary (lowest total TF-IDF weight)?
least_vocab_idx = X_tensor.sum(dim=1).argmin().item()
print(f'Paper with least vocab coverage: [{least_vocab_idx}] {df["title"][least_vocab_idx]}')

# max() returns both values and indices
values, indices = X_tensor.max(dim=1)
print('Shape of max values:  ', values.shape)
print('Shape of max indices: ', indices.shape)

Total sum (scalar):  617.5597534179688
Sum per vocab word (dim=0):  torch.Size([1000])
Sum per paper (dim=1):       torch.Size([100])
Paper with most vocab coverage: [38] Deep learning-based markerless lung tumor tracking in stereotactic radiotherapy using Siamese networks.
Paper with least vocab coverage: [40] Atmospheric-Pressure Synthesis of 2D Nitrogen-Rich Tungsten Nitride.
Shape of max values:   torch.Size([100])
Shape of max indices:  torch.Size([100])


## 5. Broadcasting

Broadcasting allows operations between tensors of different shapes by automatically expanding dimensions.

The rule: dimensions are aligned from the right, and a size-1 dimension stretches to match.

Try removing `keepdim=True` below and observe the error.

In [19]:
norms_2d = X_tensor.norm(dim=1, keepdim=True)  # shape (N, 1)
norms_1d = X_tensor.norm(dim=1)                # shape (N,)
print('norms_2d shape:', norms_2d.shape)  # (N, 1)
print('norms_1d shape:', norms_1d.shape)  # (N,)
print('X_tensor shape:', X_tensor.shape)  # (N, 1000)
# This works: (N, 1000) / (N, 1) → broadcast norms across columns
result_ok = X_tensor / norms_2d
print('Division with keepdim=True works, shape:', result_ok.shape)

# Try this without keepdim - should raise an error due to shape mismatch
#result_fail = X_tensor / norms_1d  # uncomment to test

norms_2d shape: torch.Size([100, 1])
norms_1d shape: torch.Size([100])
X_tensor shape: torch.Size([100, 1000])
Division with keepdim=True works, shape: torch.Size([100, 1000])


## 6. Batching Intuition — replace loops with matrix ops

The key mental shift in ML: instead of looping over items, operate on all of them at once.

This is faster (parallelised on CPU/GPU) and is the foundation of batched training.

In [20]:
# Loop version — find top 5 for one paper
paper_index = 0
sorted_loop = torch.argsort(sim_matrix[paper_index], descending=True)
top5_one = sorted_loop[1:6]
print('Top 5 for paper 0 (loop):  ', top5_one.tolist())
print()

# Batched version — top 5 for ALL papers at once, no loop
all_sorted = torch.argsort(sim_matrix, dim=1, descending=True)
top5_all = all_sorted[:, 1:6]  # skip column 0 (self)
print('Top 5 for ALL papers shape:', top5_all.shape)   # (N, 5)
print('Top 5 for paper 0 (batch): ', top5_all[0].tolist())  # same as above
print()
# Sanity check — both should match
print('Match:', top5_one.tolist() == top5_all[0].tolist())

Top 5 for paper 0 (loop):   [16, 1, 3, 50, 10]

Top 5 for ALL papers shape: torch.Size([100, 5])
Top 5 for paper 0 (batch):  [16, 1, 3, 50, 10]

Match: True


## 7. Einstein Summation (einsum)

`torch.einsum` is a generalised notation for tensor operations.
It's ubiquitous in ML (attention, covariance matrices, etc.).

Notation: `'ik,jk->ij'` means:
- take tensor with indices `i,k` and tensor with indices `j,k`
- sum over the shared `k` dimension
- result has indices `i,j`

This is exactly matrix multiplication `A @ B.T`.

In [21]:
sim_matmul = X_normalized @ X_normalized.T
sim_einsum = torch.einsum('ik,jk->ij', X_normalized, X_normalized)

print('Results match:', torch.allclose(sim_matmul, sim_einsum, atol=1e-5))

# einsum for row-wise dot product (diagonal of sim matrix) without computing full matrix:
self_sim = torch.einsum('ik,ik->i', X_normalized, X_normalized)  # shape (N,)
print('Self-similarities (should all be ~1.0):', self_sim[:5])

Results match: True
Self-similarities (should all be ~1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 8. Practical Exploration of Lexical Similarity

- Which two papers are most similar overall? (`sim_matrix` minus diagonal)
- Which paper is most *different* from all others? (lowest average similarity)
- What happens to similarity scores if you use `CountVectorizer` instead of `TfidfVectorizer`?

In [23]:
# Mask out the diagonal (self-similarity = 1.0) to find the most similar PAIR
sim_no_diag = sim_matrix.clone()
sim_no_diag.fill_diagonal_(0)

max_val = sim_no_diag.max().item()
max_idx = sim_no_diag.argmax().item()  # flat index as Python int
i, j = divmod(max_idx, sim_matrix.shape[1])

print(f'Most similar pair (similarity={max_val:.4f}):')
print(f'  [{i}] {df["title"].iloc[i]}')  
print(f'  [{j}] {df["title"].iloc[j]}')

Most similar pair (similarity=0.9238):
  [44] Simultaneous tumor and surrogate motion tracking with dynamic MRI for radiation therapy planning.
  [83] Evaluation of lung tumor motion management in radiation therapy with dynamic MRI.


## 9. Embedding Representation

- `CountVectorizer` produces sparse vector (a lot of zeros) with raw word counts, where each word represents one dimension
- `TfidfVectorizer` similar to the `CountVectorizer`but words that appear in many documents (IDF) are down-weighted and rare ones are up-weighted
- Embeddings produce dense vectors (no zeros) where each dimension doesn't have a particular semantic meaning, but vector geometry encodes a meaning. Cosine similarity will be high even if different words are used to convey similar message.


In [68]:
# Function to get top 5 similar papers for a given index
def top5(sim_matrix, idx):
    """Returns top 5 similar paper indices for a given paper index."""
    sims = sim_matrix[idx].clone()
    sims[idx] = 0  # mask self (cleaner than fill_diagonal_ on the whole matrix)
    return torch.argsort(sims, descending=True)[:5].tolist()  # ← .tolist() converts all at once

PAPER_INDEX = 34

In [69]:
# Generate similarity matrix from CountVectorizer and find most similar pair of papers
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", max_features=1000)
X = vectorizer.fit_transform(abstracts)
X_tensor = torch.tensor(X.toarray(), dtype=torch.float32)

norms = X_tensor.norm(dim=1, keepdim=True).clamp(min=1e-10)
X_normalized = X_tensor / norms
count_sim = X_normalized @ X_normalized.T

In [70]:
# Generate similarity matrix from TF-IDF and find most similar pair of papers

vectorizer = TfidfVectorizer(stop_words="english", max_features=1000, norm='l2')
X = vectorizer.fit_transform(abstracts)
X_tensor = torch.tensor(X.toarray(), dtype=torch.float32)

tfidf_sim = X_tensor @ X_tensor.T

In [71]:
# Generate similarity matrix from SentenceTransformer embeddings and find most similar pair of papers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
emb = torch.tensor(model.encode(abstracts))
norms = emb.norm(dim=1, keepdim=True).clamp(min=1e-8)
emb_norm = emb / norms
emb_sim = emb_norm @ emb_norm.T

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7172.61it/s]


In [72]:
print(f"Reference: [{PAPER_INDEX}] {df['title'][PAPER_INDEX]}\n")
print("CountVectorizer:"); [print(f"  {df['title'].iloc[i]}") for i in top5(count_sim, PAPER_INDEX)]
print("TF-IDF:");          [print(f"  {df['title'].iloc[i]}") for i in top5(tfidf_sim, PAPER_INDEX)]
print("Embeddings:");      [print(f"  {df['title'].iloc[i]}") for i in top5(emb_sim,   PAPER_INDEX)]

Reference: [34] Deep learning-based target decomposition for markerless lung tumor tracking in radiotherapy.

CountVectorizer:
  A Bayesian approach for three-dimensional markerless tumor tracking using kV imaging during lung radiotherapy.
  Deep learning-based markerless lung tumor tracking in stereotactic radiotherapy using Siamese networks.
  Explainable AI for raising confidence in deep learning-based tumor tracking models.
  Simultaneous tumor and surrogate motion tracking with dynamic MRI for radiation therapy planning.
  Evaluation of lung tumor motion management in radiation therapy with dynamic MRI.
TF-IDF:
  A Bayesian approach for three-dimensional markerless tumor tracking using kV imaging during lung radiotherapy.
  Deep learning-based markerless lung tumor tracking in stereotactic radiotherapy using Siamese networks.
  Explainable AI for raising confidence in deep learning-based tumor tracking models.
  Evaluation of lung tumor motion management in radiation therapy with 

[None, None, None, None, None]